## load libraries

In [1]:
import numpy as np
import os
import math

import torch.nn as nn
import torch
import torch.optim as optim

from sklearn.model_selection import train_test_split

from torch.utils.data import TensorDataset, DataLoader

from sklearn.metrics import accuracy_score


from CNN_blocks import *
from MSEL_block import *

In [2]:
device="cpu"

## load data

In [3]:
signals=np.load("../dataset/signals.npy")
labels= np.load("../dataset/labels.npy")

In [4]:
print(signals.shape)
print(labels.shape)

(10, 12, 1000)
(10, 8)


In [5]:
x=signals
y=labels

x_train, x_test, y_train, y_test = train_test_split(x, y)

mean=x_train.mean(axis=(0,2), keepdims=True)
std = x_test.std(axis=(0,2), keepdims=True)

x_train=(x_train-mean)/(std + 1e-8)
x_test =(x_test -mean)/(std + 1e-8)

x_train=torch.tensor(x_train, dtype=torch.float)
y_train=torch.tensor(y_train, dtype=torch.long)
x_test=torch.tensor(x_test, dtype=torch.float)
y_test=torch.tensor(y_test, dtype=torch.long)

In [26]:
#delete this block
tmp=x_train[-2]

In [7]:
train_dataset=TensorDataset(x_train, y_train)

In [8]:
train_loader=DataLoader(
    train_dataset,
    batch_size=5,
    shuffle=True
)

In [9]:
class Classifier(nn.Module):
    def __init__(self):
        super().__init__()

        self.CNN_blocks=CNN_blocks()

        self.lower_MSEL=lower_MSEL()
        self.upper_MSEL=upper_MSEL()
        
        self.network=nn.Sequential(
            nn.Linear(232*256, 256),
            nn.ReLU(),
            
            nn.Linear(256, 80),
            nn.ReLU(),
            
            nn.Linear(80, 8)
        )

    def forward(self, x):
        
        cnn_output=self.CNN_blocks(x)
        #net_in=cnn_output.flatten(1)
        lower_out =  self.lower_MSEL(cnn_output)
        upper_out =  self.upper_MSEL(cnn_output)
        
        net_in    =  torch.cat((lower_out, upper_out), dim=1).flatten(1)
        print(net_in.shape)
        
        return self.network( net_in )

In [10]:
model=Classifier().to(device)

In [11]:
criterian=nn.BCEWithLogitsLoss()

optimizer=optim.Adam(
    model.parameters(),
    lr=0.001
)

In [12]:
epochs=40
for epoch in range(epochs):
    model.train()
    total_loss=0
    
    for x_batch, y_batch in train_loader:
        x_batch=x_batch.to(device)
        y_batch=y_batch.to(device)

        output=model(x_batch)

        loss=criterian(output, y_batch.float())

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        total_loss+=loss.item()

        print(
            "epoch:  ", epoch, "/", epochs,"\n",
            "loss:   ", total_loss, "\n\n"
        )

torch.Size([5, 59392])


/home/rahaan/Desktop/projects/GESH-ECG-anomaly-deltector/detector_model/VE/lib/python3.10/site-packages/torch/cuda/__init__.py:422: UserWarning: Found GPU0 NVIDIA GeForce MX330 which is of compute capability (CC) 6.1.
The following list shows the CCs this version of PyTorch was built for and the hardware CCs it supports:
- 7.5 which supports hardware CC >=7.5,<8.0
- 8.0 which supports hardware CC >=8.0,<9.0 except {8.7}
- 8.6 which supports hardware CC >=8.6,<9.0 except {8.7}
- 9.0 which supports hardware CC >=9.0,<10.0
- 10.0 which supports hardware CC >=10.0,<11.0 except {10.1}
- 12.0 which supports hardware CC >=12.0,<13.0
Your installed torch==2.13.0+cu130 does not include kernels for this GPU. Reinstall the same version against a CUDA build that does, e.g.:
  For CUDA 12.6 use pip install torch==2.13.0 --index-url https://download.pytorch.org/whl/cu126
  _warn_unsupported_code(d, device_cc, code_ccs)
/home/rahaan/Desktop/projects/GESH-ECG-anomaly-deltector/detector_model/VE/lib/py

epoch:   0 / 40 
 loss:    0.7098881602287292 


torch.Size([2, 59392])
epoch:   0 / 40 
 loss:    1.0142396986484528 


torch.Size([5, 59392])
epoch:   1 / 40 
 loss:    0.4906229078769684 


torch.Size([2, 59392])
epoch:   1 / 40 
 loss:    0.5751778036355972 


torch.Size([5, 59392])
epoch:   2 / 40 
 loss:    0.596086859703064 


torch.Size([2, 59392])
epoch:   2 / 40 
 loss:    1.3874343037605286 


torch.Size([5, 59392])
epoch:   3 / 40 
 loss:    0.544253945350647 


torch.Size([2, 59392])
epoch:   3 / 40 
 loss:    0.6791702806949615 


torch.Size([5, 59392])
epoch:   4 / 40 
 loss:    0.32936739921569824 


torch.Size([2, 59392])
epoch:   4 / 40 
 loss:    0.8282532691955566 


torch.Size([5, 59392])
epoch:   5 / 40 
 loss:    0.16742616891860962 


torch.Size([2, 59392])
epoch:   5 / 40 
 loss:    0.5520816743373871 


torch.Size([5, 59392])
epoch:   6 / 40 
 loss:    0.024735290557146072 


torch.Size([2, 59392])
epoch:   6 / 40 
 loss:    0.0370632391422987 


torch.Size([5

In [29]:
x_test=x_test.to(device)
model.eval()
with torch.no_grad():
    y_pred=model(x_test)
print(y_pred.round())

torch.Size([3, 59392])
tensor([[-53., -22., -30., -32.,  15., -16., -12., -24.],
        [-51., -22., -29., -32.,  15., -15., -12., -23.],
        [-51., -21., -28., -31.,  14., -13., -14., -23.]])


In [14]:
y_test

tensor([[0, 0, 0, 0, 1, 0, 0, 0],
        [0, 0, 0, 0, 1, 0, 0, 0],
        [0, 0, 0, 0, 1, 0, 0, 0]])

In [28]:
tmp=x_train
tmp=tmp.to(device)
model.eval()
with torch.no_grad():
    y_pred=model(tmp)
print(y_pred.round())

torch.Size([7, 59392])
tensor([[-56., -25., -28., -32.,  12., -17., -22., -26.],
        [-57., -25., -30., -34.,  14., -17., -19., -26.],
        [-48., -20., -32., -31.,  19., -24.,   9., -22.],
        [-55., -24., -28., -33.,  11., -16., -22., -25.],
        [-51., -23., -27., -30.,  12., -17., -17., -24.],
        [-40., -12., -26., -26.,  21.,   7.,   6., -19.],
        [-56., -24., -30., -31.,  15., -16., -17., -26.]])


In [17]:
y_train

tensor([[0, 0, 0, 0, 1, 0, 0, 0],
        [0, 0, 0, 0, 1, 0, 0, 0],
        [0, 0, 0, 0, 1, 0, 1, 0],
        [0, 0, 0, 0, 1, 0, 0, 0],
        [0, 0, 0, 0, 1, 0, 0, 0],
        [0, 0, 0, 0, 1, 1, 1, 0],
        [0, 0, 0, 0, 1, 0, 0, 0]])